# r2x spike S3: stand-in framework run of a process
Throwaway. Runs as the stand-in framework account. Reads `process.yml` at a fixed commit, checks its structure, records a request, passes plain values into every step, and submits one one-time run as the client account with one task per step. `after` becomes the task's "runs after" links, so Databricks runs independent steps at the same time. Optionally repairs a failed run once.

In [0]:
dbutils.widgets.text("test_name", "s3")
dbutils.widgets.text("git_url", "https://github.com/m-raeber/databricks_framework_v2_spikes")
dbutils.widgets.text("git_commit", "")
dbutils.widgets.text("process_dir", "process_s3")
dbutils.widgets.text("client_sp", "9b365b7d-97fc-45ac-9d98-92fc1d6e723c")
dbutils.widgets.text("warehouse_id", "c1dfcbeb7d45a146")
dbutils.widgets.text("viewer", "matt.raeber@integrate-solutions.com")
dbutils.widgets.text("settings_json", "{}")      # the request's setting values, e.g. {"fail_step": "totals"}
dbutils.widgets.text("repair_with_json", "")     # if set and the run fails: repair once with these setting values
names = ["test_name", "git_url", "git_commit", "process_dir", "client_sp", "warehouse_id", "viewer", "settings_json", "repair_with_json"]
p = {k: dbutils.widgets.get(k) for k in names}
assert p["git_commit"], "git_commit is required"

In [0]:
import json, os, urllib.request, uuid, yaml

raw_base = p["git_url"].replace("https://github.com/", "https://raw.githubusercontent.com/") + f"/{p['git_commit']}/{p['process_dir']}"
spec = yaml.safe_load(urllib.request.urlopen(f"{raw_base}/process.yml", timeout=30).read().decode())

problems = []
if spec.get("format") != 1:
    problems.append(f"format {spec.get('format')!r} not known")
steps = spec.get("steps") or []
step_names = [s.get("name") for s in steps]
if len(set(step_names)) != len(step_names):
    problems.append(f"step names not unique: {step_names}")
for s in steps:
    if sum(k in s for k in ("notebook", "sql")) != 1:
        problems.append(f"step {s.get('name')}: give exactly one of notebook or sql")
    problems += [f"step {s['name']}: after names unknown step {a}" for a in s.get("after") or [] if a not in step_names]
    problems += [f"step {s['name']}: writes unknown output {o}" for o in s.get("writes") or [] if o not in (spec.get("outputs") or {})]

# no loops: repeatedly take steps whose `after` steps are all done
after = {s["name"]: set(s.get("after") or []) for s in steps}
left = dict(after)
while left:
    ready = [n for n, d in left.items() if not d & left.keys()]
    if not ready:
        problems.append(f"loop between steps {sorted(left)}")
        break
    for n in ready:
        del left[n]

# steps that can run at the same time must not write the same output
def before(n, seen=None):
    seen = set() if seen is None else seen
    for a in after.get(n, ()):
        if a not in seen:
            seen.add(a); before(a, seen)
    return seen
writes = {s["name"]: set(s.get("writes") or []) for s in steps}
for i, a in enumerate(step_names):
    for b in step_names[i + 1:]:
        if a not in before(b) and b not in before(a) and writes.get(a, set()) & writes.get(b, set()):
            problems.append(f"steps {a} and {b} can run at the same time and both write {sorted(writes[a] & writes[b])}")

# versions: only in requirements.txt; every line must be an exact version or a file in the libraries volume
import re
LIB_VOLUME = "/Volumes/sconset-db-01/r2x_spike/libraries/"
req_lines = [l.strip() for l in urllib.request.urlopen(f"{raw_base}/requirements.txt", timeout=30).read().decode().splitlines()
             if l.strip() and not l.strip().startswith("#")]
versions = {}
for l in req_lines:
    if l.startswith(LIB_VOLUME):
        m = re.match(r"(\w+)-([\w.]+)-py3-none-any\.whl$", l[len(LIB_VOLUME):])
        if not m or not os.path.exists(l):
            problems.append(f"requirements.txt: {l} is not an install file in the libraries volume")
        else:
            versions[m.group(1)] = m.group(2)
    elif re.match(r"^[A-Za-z0-9_.\-]+==[\w.]+$", l):
        name, v = l.split("=="); versions[name.lower()] = v
    else:
        problems.append(f"requirements.txt: {l} has no exact version")
if "r2x_helper" not in versions:
    problems.append("requirements.txt does not name a helper version")

print("structure check:", "OK" if not problems else problems)
assert not problems, problems

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

T = "`sconset-db-01`.r2x_spike"
# Spike only: where this process's declared inputs and outputs live in DEV
inputs_map = {"sales": f"{T}.input_sample"}
outputs_map = {"totals": f"{T}.staging_out", "counts": f"{T}.staging_other"}

w = WorkspaceClient()
who = spark.sql("SELECT current_user()").first()[0]
request_id = f"r2x-{p['test_name']}-{uuid.uuid4().hex[:8]}"

requested = json.loads(p["settings_json"] or "{}")
unknown = set(requested) - set(spec.get("settings") or {})
assert not unknown, f"settings not declared in process.yml: {sorted(unknown)}"

values = {"request_id": request_id}
for n in spec.get("inputs") or {}:
    values[f"input_{n}"] = inputs_map[n]
    values[f"input_{n}_version"] = str(spark.sql(f"DESCRIBE HISTORY {inputs_map[n]} LIMIT 1").first()["version"])
for n in spec.get("outputs") or {}:
    values[f"output_{n}"] = outputs_map[n]
for n, default in (spec.get("settings") or {}).items():
    values[f"setting_{n}"] = str(requested.get(n, default))

spark.sql(f"INSERT INTO {T}.request VALUES (:rid, :t, :who, current_timestamp(), NULL, NULL)",
          args={"rid": request_id, "t": p["test_name"], "who": who})
print(json.dumps(values, indent=2))

In [0]:
def task_for(s):
    path = f"{p['process_dir']}/{s.get('notebook') or s.get('sql')}"
    if "notebook" in s:
        work = dict(notebook_task=jobs.NotebookTask(notebook_path=path, source=jobs.Source.GIT, base_parameters=values))
    else:
        work = dict(sql_task=jobs.SqlTask(file=jobs.SqlTaskFile(path=path, source=jobs.Source.GIT),
                                          warehouse_id=p["warehouse_id"], parameters=values))
    return jobs.SubmitTask(task_key=s["name"],
                           depends_on=[jobs.TaskDependency(task_key=a) for a in s.get("after") or []] or None,
                           disable_auto_optimization=True,  # no automatic retries; failures stay visible
                           **work)

waiter = w.jobs.submit(
    run_name=f"r2x-spike-{p['test_name']}-{request_id[-8:]}",
    idempotency_token=request_id,
    run_as=jobs.JobRunAs(service_principal_name=p["client_sp"]),
    git_source=jobs.GitSource(git_url=p["git_url"], git_provider=jobs.GitProvider.GIT_HUB, git_commit=p["git_commit"]),
    access_control_list=[jobs.JobAccessControlRequest(user_name=p["viewer"], permission_level=jobs.JobPermissionLevel.CAN_VIEW)] if p["viewer"] else None,
    tasks=[task_for(s) for s in steps],
)
client_run_id = waiter.run_id
spark.sql(f"UPDATE {T}.request SET client_run_id = :c WHERE request_id = :rid", args={"c": client_run_id, "rid": request_id})
try:
    waiter.result()
except Exception as e:
    print("run did not succeed:", str(e)[:300])

In [0]:
first = w.jobs.get_run(client_run_id)
repair_id = None
if str(first.state.result_state) != "RunResultState.SUCCESS" and p["repair_with_json"]:
    fixed = {f"setting_{k}": str(v) for k, v in json.loads(p["repair_with_json"]).items()}
    rw = w.jobs.repair_run(run_id=client_run_id, rerun_all_failed_tasks=True, rerun_dependent_tasks=True,
                           notebook_params=fixed)
    repair_id = rw.response.repair_id if hasattr(rw, "response") and rw.response else None
    try:
        rw.result()
    except Exception as e:
        print("repair did not succeed:", str(e)[:300])

In [0]:
run = w.jobs.get_run(client_run_id, include_history=True)
timeline = []
for t in sorted(run.tasks, key=lambda t: (t.start_time or 0)):
    try:
        o = w.jobs.get_run_output(t.run_id)
        result = o.notebook_output.result if o.notebook_output else None
        error = o.error
    except Exception as e:
        result, error = None, str(e)[:200]
    timeline.append({"step": t.task_key, "task_run_id": t.run_id, "attempt": t.attempt_number,
                     "state": str(t.state.result_state), "start": t.start_time, "end": t.end_time,
                     "used_commit": t.git_source.git_snapshot.used_commit if t.git_source and t.git_source.git_snapshot else None,
                     "result": result, "error": (error or "")[:200]})
out = {"framework_ran_as": who, "request_id": request_id, "client_run_id": client_run_id, "repair_id": repair_id,
       "first_state": str(first.state.result_state), "final_state": str(run.state.result_state),
       "commit_requested": p["git_commit"], "versions": versions,
       "values": values, "repair_history": [{"type": str(r.type), "id": r.id, "task_run_ids": r.task_run_ids} for r in (run.repair_history or [])],
       "timeline": timeline}
print(json.dumps(out, indent=2, default=str))
dbutils.notebook.exit(json.dumps(out, default=str))